# Ноутбук A/B эксперимента (Lab 4)

Сравнение промптов v1 (Baseline) и v2 (Итерация).
Результаты выгружаются в `ab_test_results.json`.

In [ ]:
!pip install openai tenacity pandas -q
import os
import json
import time
import pandas as pd
from openai import OpenAI
from tenacity import retry, stop_after_attempt, wait_exponential

# Ключ нужно подставлять локально при запуске (мы его удалили для безопасности)
# os.environ["OPENROUTER_API_KEY"] = "sk-or-v1-..."

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ.get("OPENROUTER_API_KEY", "sk-placeholder"))
MODEL = "openrouter/free"

# Импортируем промпты (предполагается, что они лежат в ml/prompts_v2/prompts.py)
import sys
sys.path.append('.')
try:
    from ml.prompts_v2.prompts import prompt_summarize_v1, prompt_summarize_v2, prompt_classify_v1, prompt_classify_v2
except ImportError:
    print("ВНИМАНИЕ: Не удалось импортировать prompts_v2. Убедитесь, что файл на месте.")

@retry(stop=stop_after_attempt(2), wait=wait_exponential(multiplier=1, min=2, max=5))
def call_llm(prompt: str):
    start = time.time()
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"},
        )
        content = response.choices[0].message.content
        latency = time.time() - start
        return content, latency, response.usage.total_tokens
    except Exception as e:
        return None, 0.0, 0

In [ ]:
games_data = [
    {"game_id": "nr_002", "store_description": "Командный шутер 3v3 с короткими матчами. Сезонные пропуска.", "metrics_4w": [100, 120, 140, 160]},
    {"game_id": "tt_001", "store_description": "Merge-головоломка. Восстанавливайте таверну.", "metrics_4w": [500, 600, 700, 800]},
    {"game_id": "so_003", "store_description": "Farm-симулятор. Выращивайте сад, собирайте урожай.", "metrics_4w": [100, 95, 90, 200]},
    {"game_id": "synth_004", "store_description": "Космический idle-симулятор с PvP.", "metrics_4w": [300, 280, 260, 500]},
    {"game_id": "synth_005", "store_description": "Коллекционная карточная игра с пошаговыми боями.", "metrics_4w": [150, 140, 130, 280]},
]

results = []
print("Запуск A/B теста...")
for game in games_data:
    gid = game["game_id"]
    
    # v1
    p_sum_v1 = prompt_summarize_v1(gid, game["store_description"])
    ans_sum_v1, lat_sum_v1, tok_sum_v1 = call_llm(p_sum_v1)
    p_cls_v1 = prompt_classify_v1(gid, game["metrics_4w"])
    ans_cls_v1, lat_cls_v1, tok_cls_v1 = call_llm(p_cls_v1)
    
    # v2
    p_sum_v2 = prompt_summarize_v2(gid, game["store_description"])
    ans_sum_v2, lat_sum_v2, tok_sum_v2 = call_llm(p_sum_v2)
    p_cls_v2 = prompt_classify_v2(gid, game["metrics_4w"])
    ans_cls_v2, lat_cls_v2, tok_cls_v2 = call_llm(p_cls_v2)

    results.append({
        "game_id": gid,
        "v1_narrative": ans_sum_v1 if ans_sum_v1 else "ERROR",
        "v1_category": ans_cls_v1 if ans_cls_v1 else "ERROR",
        "v1_tokens": tok_sum_v1 + tok_cls_v1,
        "v1_latency": round(lat_sum_v1 + lat_cls_v1, 2),
        "v2_narrative": ans_sum_v2 if ans_sum_v2 else "ERROR",
        "v2_category": ans_cls_v2 if ans_cls_v2 else "ERROR",
        "v2_tokens": tok_sum_v2 + tok_cls_v2,
        "v2_latency": round(lat_sum_v2 + lat_cls_v2, 2),
    })

df = pd.DataFrame(results)
df.to_csv("ab_test_results.csv", index=False)
df.to_json("ab_test_results.json", orient="records", force_ascii=False)
display(df)
print("Результаты сохранены в ab_test_results.json")